# 전처리 다시...

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

BASE_PATH = "/content/drive/MyDrive/wafer-map-defect-xai"
print(os.path.exists(BASE_PATH))
print(os.listdir("/content/drive/MyDrive"))

True
['Google 어스', 'IMG_8762.jpg', 'Colab Notebooks', '자료구조', '2학년 압축.zip', 'endLoop()_기능명세서.gsheet', 'openCV_코드_분석_최종.pdf', '블러링_블렌딩_스레드홀딩정리.pdf', '문제와개념_컴비.pdf', '13팀_빠니보틀.zip', '1과목_준호의정리.pdf', 'UXUI.zip', '서비스기획.gdoc', '2025_Proposal_팀카페인.docx', '2025_Proposal_팀카페인.gdoc', '디지털마케팅_중간고사_대비(1~2장).pdf', '디지털마케팅_중간고사_대비(3장_4장).pdf', '발표대폰_구조상ㅁ부터.pdf', 'ML_항공기결항예측모델.pdf', 'studying_Coding', 'TEMI 로봇 사용성 평가 질문.gform', '제목 없는 설문지.gform', 'TEMI 로봇 사용성 평가 질문(응답).gsheet', '영수증.gdoc', 'Capstone Kick-Off.gslides', '킥오프발표자료.zip', '포폴관련', 'ko_KR.parquet', '[팀_자리비움]_산학연계AI프로젝트_과제제안서.pdf', '1201기준 결과.gsheet', 'wafer-map-defect-xai']


In [3]:
import os

BASE_PATH = "/content/drive/MyDrive/wafer-map-defect-xai"

for root, dirs, files in os.walk(BASE_PATH):
    level = root.replace(BASE_PATH, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files[:10]:
        print(f"{subindent}{file}")

wafer-map-defect-xai/
  data/
    raw/
      LSWMD.pkl
    processed/
      X_val.npy
      X_test.npy
      y_train.npy
      y_val.npy
      y_test.npy
      classes.npy
  outputs/
    figures/
    models/
    reports/
  notebooks/
    wafer_map_defect_xai.ipynb
    01_data_eda_preprocessing.ipynb


In [4]:
import os
import pandas as pd

RAW_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/raw/LSWMD.pkl"

print("RAW_PATH exists:", os.path.exists(RAW_PATH))

df = pd.read_pickle(RAW_PATH)

print(df.shape)
df.head()

RAW_PATH exists: True
(811457, 6)


,waferMap,dieSize,lotName,waferIndex,trianTestLabel,failureType
0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,1.0,[[Training]],[[none]]
1,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,2.0,[[Training]],[[none]]
2,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,3.0,[[Training]],[[none]]
3,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,4.0,[[Training]],[[none]]
4,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,5.0,[[Training]],[[none]]


# 라벨이 있는 데이터만 확인하기

In [5]:
import numpy as np

def unwrap_label(x):
    try:
        return x[0][0]
    except:
        return None

df["failure_label"] = df["failureType"].apply(unwrap_label)
df["train_test_label"] = df["trianTestLabel"].apply(unwrap_label)

print("failure label counts:")
print(df["failure_label"].value_counts(dropna=False))

print("\ntrain/test label counts:")
print(df["train_test_label"].value_counts(dropna=False))

failure label counts:
failure_label
None         638507
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64

train/test label counts:
train_test_label
None        638507
Test        118595
Training     54355
Name: count, dtype: int64


# 라벨 있는 데이터만 필터링

In [6]:
labeled_df = df[df["failure_label"].notna()].copy()

print("labeled_df shape:", labeled_df.shape)
print(labeled_df["failure_label"].value_counts())
print()
print(labeled_df["train_test_label"].value_counts())


labeled_df shape: (172950, 8)
failure_label
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64

train_test_label
Test        118595
Training     54355
Name: count, dtype: int64


In [7]:
import numpy as np

CLASSES_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/processed/classes.npy"

classes = np.load(CLASSES_PATH, allow_pickle=True)
print(classes)

[np.str_('Center') np.str_('Donut') np.str_('Edge-Loc')
 np.str_('Edge-Ring') np.str_('Loc') np.str_('Near-full')
 np.str_('Random') np.str_('Scratch') np.str_('none')]


# Training 데이터만 분리하기

In [8]:
train_df = labeled_df[labeled_df["train_test_label"] == "Training"].copy()
test_df = labeled_df[labeled_df["train_test_label"] == "Test"].copy()

print("train_df:", train_df.shape)
print("test_df:", test_df.shape)

print(train_df["failure_label"].value_counts())

train_df: (54355, 8)
test_df: (118595, 8)
failure_label
none         36730
Edge-Ring     8554
Center        3462
Edge-Loc      2417
Loc           1620
Random         609
Scratch        500
Donut          409
Near-full       54
Name: count, dtype: int64


## waferMap 크기 확인

In [9]:
import numpy as np

sample_map = train_df["waferMap"].iloc[0]
sample_map = np.array(sample_map)

print(type(sample_map))
print(sample_map.shape)
print(sample_map)

<class 'numpy.ndarray'>
(45, 48)
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


### 전체 wafer map 크기들이 얼마나 다양한지

In [10]:
shape_counts = train_df["waferMap"].apply(lambda x: np.array(x).shape).value_counts()

print(shape_counts.head(20))
print("unique shapes:", len(shape_counts))

waferMap
(25, 27)      15684
(27, 25)       9235
(26, 26)       6369
(38, 36)       1877
(33, 37)       1804
(53, 52)       1519
(63, 62)       1424
(45, 43)       1035
(48, 48)        751
(41, 41)        514
(69, 73)        484
(107, 150)      470
(46, 46)        454
(39, 31)        454
(38, 38)        436
(41, 42)        431
(50, 43)        423
(32, 29)        413
(35, 40)        408
(25, 26)        330
Name: count, dtype: int64
unique shapes: 335


# X_train 생성 및 저장

In [11]:
import os
import numpy as np
from skimage.transform import resize
from tqdm import tqdm

PROCESSED_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/processed"
X_TRAIN_PATH = os.path.join(PROCESSED_PATH, "X_train.npy")

IMG_SIZE = 64

def resize_wafer_map(wafer_map, img_size=64):
    wafer_map = np.array(wafer_map)

    resized = resize(
        wafer_map,
        (img_size, img_size),
        order=0,              # nearest-neighbor: 0,1,2 값 유지에 유리
        preserve_range=True,
        anti_aliasing=False
    )

    return resized.astype(np.uint8)

X_train = []

for wm in tqdm(train_df["waferMap"], desc="Creating X_train"):
    X_train.append(resize_wafer_map(wm, IMG_SIZE))

X_train = np.array(X_train, dtype=np.uint8)

# CNN 입력 형태로 channel dimension 추가: (N, 64, 64, 1)
X_train = np.expand_dims(X_train, axis=-1)

print("X_train shape:", X_train.shape)
print("X_train dtype:", X_train.dtype)
print("unique values:", np.unique(X_train))

np.save(X_TRAIN_PATH, X_train)

print("saved:", X_TRAIN_PATH)
print("exists:", os.path.exists(X_TRAIN_PATH))

Creating X_train: 100%|██████████| 54355/54355 [00:20<00:00, 2591.26it/s]


X_train shape: (54355, 64, 64, 1)
X_train dtype: uint8
unique values: [0 1 2]
saved: /content/drive/MyDrive/wafer-map-defect-xai/data/processed/X_train.npy
exists: True


# 전체 processed 파일 최종 확인

In [12]:
import os

PROCESSED_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/processed"

print(os.listdir(PROCESSED_PATH))

['X_val.npy', 'X_test.npy', 'y_train.npy', 'y_val.npy', 'y_test.npy', 'classes.npy', 'X_train.npy']


## 각 데이터 shape도 확인

In [13]:
import numpy as np
import os

PROCESSED_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/processed"

X_train = np.load(os.path.join(PROCESSED_PATH, "X_train.npy"))
X_val = np.load(os.path.join(PROCESSED_PATH, "X_val.npy"))
X_test = np.load(os.path.join(PROCESSED_PATH, "X_test.npy"))

y_train = np.load(os.path.join(PROCESSED_PATH, "y_train.npy"))
y_val = np.load(os.path.join(PROCESSED_PATH, "y_val.npy"))
y_test = np.load(os.path.join(PROCESSED_PATH, "y_test.npy"))

classes = np.load(os.path.join(PROCESSED_PATH, "classes.npy"), allow_pickle=True)

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:", X_val.shape, X_val.dtype)
print("X_test:", X_test.shape, X_test.dtype)

print("y_train:", y_train.shape, y_train.dtype)
print("y_val:", y_val.shape, y_val.dtype)
print("y_test:", y_test.shape, y_test.dtype)

print("classes:", classes)

X_train: (54355, 64, 64, 1) uint8
X_val: (25942, 64, 64, 1) float32
X_test: (25943, 64, 64, 1) float32
y_train: (121065,) int64
y_val: (25942,) int64
y_test: (25943,) int64
classes: [np.str_('Center') np.str_('Donut') np.str_('Edge-Loc')
 np.str_('Edge-Ring') np.str_('Loc') np.str_('Near-full')
 np.str_('Random') np.str_('Scratch') np.str_('none')]


X_train: (54355, 64, 64, 1)
y_train: (121065,)
이 둘이 지금 개수가 안맞음...
이미지는 54,355장인데 정답 라벨은 121,065개라고?

# 기존 processed 파일 백업하기

In [14]:
import os
import shutil

PROCESSED_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/processed"
BACKUP_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/processed_backup"

os.makedirs(BACKUP_PATH, exist_ok=True)

for file in os.listdir(PROCESSED_PATH):
    src = os.path.join(PROCESSED_PATH, file)
    dst = os.path.join(BACKUP_PATH, file)
    shutil.copy2(src, dst)

print("backup done")
print(os.listdir(BACKUP_PATH))

backup done
['X_val.npy', 'X_test.npy', 'y_train.npy', 'y_val.npy', 'y_test.npy', 'classes.npy', 'X_train.npy']


In [15]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from skimage.transform import resize
from tqdm import tqdm

PROCESSED_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/processed"

IMG_SIZE = 64
RANDOM_STATE = 42

classes = np.array([
    "Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc",
    "Near-full", "Random", "Scratch", "none"
])

class_to_idx = {label: idx for idx, label in enumerate(classes)}

# labeled_df는 이미 만들어져 있어야 함
work_df = labeled_df.copy()
work_df["label_idx"] = work_df["failure_label"].map(class_to_idx)

print("work_df:", work_df.shape)
print(work_df["failure_label"].value_counts())
print(work_df["label_idx"].isna().sum())

work_df: (172950, 9)
failure_label
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64
0


In [16]:
train_df, temp_df = train_test_split(
    work_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=work_df["label_idx"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label_idx"]
)

print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("test_df:", test_df.shape)

print("\ntrain label counts:")
print(train_df["failure_label"].value_counts())

print("\nval label counts:")
print(val_df["failure_label"].value_counts())

print("\ntest label counts:")
print(test_df["failure_label"].value_counts())

train_df: (121065, 9)
val_df: (25942, 9)
test_df: (25943, 9)

train label counts:
failure_label
none         103202
Edge-Ring      6776
Edge-Loc       3632
Center         3006
Loc            2515
Scratch         835
Random          606
Donut           389
Near-full       104
Name: count, dtype: int64

val label counts:
failure_label
none         22114
Edge-Ring     1452
Edge-Loc       778
Center         644
Loc            539
Scratch        179
Random         130
Donut           83
Near-full       23
Name: count, dtype: int64

test label counts:
failure_label
none         22115
Edge-Ring     1452
Edge-Loc       779
Center         644
Loc            539
Scratch        179
Random         130
Donut           83
Near-full       22
Name: count, dtype: int64


In [17]:
import os
import numpy as np
from skimage.transform import resize
from tqdm import tqdm

PROCESSED_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/processed"
IMG_SIZE = 64

def resize_wafer_map(wafer_map, img_size=64):
    wafer_map = np.array(wafer_map)

    resized = resize(
        wafer_map,
        (img_size, img_size),
        order=0,
        preserve_range=True,
        anti_aliasing=False
    )

    return resized.astype(np.uint8)

def make_X(df_part, name):
    X = []

    for wm in tqdm(df_part["waferMap"], desc=f"Creating {name}"):
        X.append(resize_wafer_map(wm, IMG_SIZE))

    X = np.array(X, dtype=np.uint8)
    X = np.expand_dims(X, axis=-1)

    print(f"{name} shape:", X.shape)
    print(f"{name} dtype:", X.dtype)
    print(f"{name} unique values:", np.unique(X))

    return X

def make_y(df_part, name):
    y = df_part["label_idx"].values.astype(np.int64)

    print(f"{name} shape:", y.shape)
    print(f"{name} dtype:", y.dtype)
    print(f"{name} unique labels:", np.unique(y))

    return y

X_train = make_X(train_df, "X_train")
X_val = make_X(val_df, "X_val")
X_test = make_X(test_df, "X_test")

y_train = make_y(train_df, "y_train")
y_val = make_y(val_df, "y_val")
y_test = make_y(test_df, "y_test")

np.save(os.path.join(PROCESSED_PATH, "X_train.npy"), X_train)
np.save(os.path.join(PROCESSED_PATH, "X_val.npy"), X_val)
np.save(os.path.join(PROCESSED_PATH, "X_test.npy"), X_test)

np.save(os.path.join(PROCESSED_PATH, "y_train.npy"), y_train)
np.save(os.path.join(PROCESSED_PATH, "y_val.npy"), y_val)
np.save(os.path.join(PROCESSED_PATH, "y_test.npy"), y_test)

np.save(os.path.join(PROCESSED_PATH, "classes.npy"), classes)

print("All processed files saved successfully.")
print(os.listdir(PROCESSED_PATH))

Creating X_train: 100%|██████████| 121065/121065 [00:29<00:00, 4163.08it/s]


X_train shape: (121065, 64, 64, 1)
X_train dtype: uint8
X_train unique values: [0 1 2]


Creating X_val: 100%|██████████| 25942/25942 [00:05<00:00, 4800.32it/s]


X_val shape: (25942, 64, 64, 1)
X_val dtype: uint8
X_val unique values: [0 1 2]


Creating X_test: 100%|██████████| 25943/25943 [00:03<00:00, 6896.82it/s]


X_test shape: (25943, 64, 64, 1)
X_test dtype: uint8
X_test unique values: [0 1 2]
y_train shape: (121065,)
y_train dtype: int64
y_train unique labels: [0 1 2 3 4 5 6 7 8]
y_val shape: (25942,)
y_val dtype: int64
y_val unique labels: [0 1 2 3 4 5 6 7 8]
y_test shape: (25943,)
y_test dtype: int64
y_test unique labels: [0 1 2 3 4 5 6 7 8]
All processed files saved successfully.
['X_val.npy', 'X_test.npy', 'y_train.npy', 'y_val.npy', 'y_test.npy', 'classes.npy', 'X_train.npy']


In [18]:
import os
import numpy as np

PROCESSED_PATH = "/content/drive/MyDrive/wafer-map-defect-xai/data/processed"

X_train = np.load(os.path.join(PROCESSED_PATH, "X_train.npy"))
X_val = np.load(os.path.join(PROCESSED_PATH, "X_val.npy"))
X_test = np.load(os.path.join(PROCESSED_PATH, "X_test.npy"))

y_train = np.load(os.path.join(PROCESSED_PATH, "y_train.npy"))
y_val = np.load(os.path.join(PROCESSED_PATH, "y_val.npy"))
y_test = np.load(os.path.join(PROCESSED_PATH, "y_test.npy"))

classes = np.load(os.path.join(PROCESSED_PATH, "classes.npy"), allow_pickle=True)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)

print("X_val:", X_val.shape, X_val.dtype)
print("y_val:", y_val.shape, y_val.dtype)

print("X_test:", X_test.shape, X_test.dtype)
print("y_test:", y_test.shape, y_test.dtype)

print("classes:", classes)

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

assert X_train.shape[1:] == (64, 64, 1)
assert X_val.shape[1:] == (64, 64, 1)
assert X_test.shape[1:] == (64, 64, 1)

print("검증 완료: X/y 개수와 이미지 shape 모두 정상입니다.")

X_train: (121065, 64, 64, 1) uint8
y_train: (121065,) int64
X_val: (25942, 64, 64, 1) uint8
y_val: (25942,) int64
X_test: (25943, 64, 64, 1) uint8
y_test: (25943,) int64
classes: ['Center' 'Donut' 'Edge-Loc' 'Edge-Ring' 'Loc' 'Near-full' 'Random'
 'Scratch' 'none']
검증 완료: X/y 개수와 이미지 shape 모두 정상입니다.
